In [215]:
import numpy as np
import pandas as pd

INPUT_ID = 0
trans_df = pd.read_csv(f"./datasets/input_{INPUT_ID}.csv")
print("SIZE:", trans_df.size)
trans_df.head(5)

SIZE: 1100000


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/08 00:30,137946,819563190,24950,8199CC620,66.76,US Dollar,66.76,US Dollar,Wire,0
1,2022/09/01 19:59,70,10042B738,215620,8078C6940,137400.35,Yen,137400.35,Yen,Credit Card,0
2,2022/09/07 18:52,70,10042B660,1120,804505360,181.62,US Dollar,181.62,US Dollar,Cheque,0
3,2022/09/01 13:23,14648,8065BE960,14648,8065BE960,36491.09,US Dollar,36491.09,US Dollar,Reinvestment,0
4,2022/09/06 14:26,13590,8058AC3B0,130593,80BEA6940,249.37,Euro,249.37,Euro,Cheque,0


In [216]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/12 21:16]


In [217]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Series([], Name: Bank, dtype: int64)

In [218]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 36784


In [219]:
# Analyze accounts.
accounts_df = pd.read_csv(f"./datasets/accounts_{INPUT_ID}.csv")
print("SIZE:", accounts_df.shape[0])

SIZE: 69627


In [220]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 20115


In [221]:
ranged_trans_usd_sept_df = trans_usd_sept_1st_df\
    .groupby(["From Bank", "Account"])\
    .filter(lambda x: x.groupby(["To Bank", "Account.1"]).size().size > 5)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 1567


In [222]:
#1. Amount, source and target accounts for transactions of less than 50 USD.

low_profile_transactions = trans_usd_df[trans_usd_df['Amount Paid'] < 50]
low_profile_transactions = low_profile_transactions[['From Bank', 'Account', 'To Bank','Account.1', 'Amount Paid']]
low_profile_transactions.sort_values(by=["From Bank"], ascending=True)

,From Bank,Account,To Bank,Account.1,Amount Paid
169,1,800544A40,18990,80D91D5B0,34.73
65430,1,8007BB640,212996,806254DF0,8.70
67740,1,80699F1F0,15786,80DB8D030,15.53
41674,1,80037C5D0,1217,800713E40,22.40
91119,1,80132E300,215514,8082ED1F0,33.54
...,...,...,...,...,...
44217,366643,818651DB0,128019,81861A990,47.92
31707,366670,8186871F0,64156,8186868C0,47.62
65060,373066,81AC48850,14595,81AC4D9A0,46.85
49096,374791,81B5CED40,160909,81B5FCC70,48.03


In [223]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
max_amount_bank = max_amount_trans_usd.merge(accounts_df, left_on="From Bank", right_on="Bank ID")
max_amount_bank=max_amount_bank[["From Bank", "Account", "Bank Name","Amount Paid"]].drop_duplicates().sort_values(by="Account", ascending=True)
max_amount_bank

,From Bank,Account,Bank Name,Amount Paid
6088,70,10042B660,Willows Thrift,1.693065e+09
1263,7,80009A720,UK Bank #19,7.492979e+04
14784,3209,800103C20,Bank of Pittsburgh,1.636080e+04
14787,3234,80013F0D0,National Bank of Springfield,1.324790e+04
7016,224,8001602D0,National Bank of Seattle,1.855508e+07
...,...,...,...,...
34878,33361,81BF96680,National Bank of Los Angeles,8.523360e+03
44529,376564,81BFA2D80,Savings Bank of Harrisburg,1.094189e+04
43477,260472,81BFBD020,National Bank of Plattsburg,3.294295e+05
44530,376610,81BFE18F0,First Bank of Albany,2.207490e+03


In [224]:
#3. Source account, payment format, and amount of transactions in period [2022-09-06, 2022-11-06] with amount lower than AVG/100 of period [2022-09-01, 2022-09-05] for the same type of transaction.

avg_amounts_per_type = trans_usd_sept_1st_df.groupby(["Payment Format"])["Amount Paid"].mean().reset_index()
trans_usd_sept_2nd_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/06') & (trans_usd_df["Timestamp"] <= '2022/09/15')]
trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_df.merge(avg_amounts_per_type, left_on=["Payment Format"], right_on=["Payment Format"]).rename(columns={
    "Amount Paid_x": "Amount Paid",
    "Amount Paid_y": "AVG",
})

print(trans_usd_sept_2nd_with_avg_df.loc[:, ["AVG", "Payment Format"]].drop_duplicates())
lower_trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_with_avg_df[trans_usd_sept_2nd_with_avg_df["Amount Paid"] < trans_usd_sept_2nd_with_avg_df["AVG"] * 0.01]
lower_trans_usd_sept_2nd_with_avg_df=lower_trans_usd_sept_2nd_with_avg_df[["From Bank", "Account", "Payment Format", "Amount Paid"]].sort_values(by=["Account", "Amount Paid"], ascending=True)
lower_trans_usd_sept_2nd_with_avg_df

             AVG Payment Format
0  186981.915358           Wire
1  630991.216033         Cheque
2    3355.320520    Credit Card
3  532452.856858            ACH
8  472532.542862           Cash


,From Bank,Account,Payment Format,Amount Paid
1143,70,10042B660,Cash,0.21
12649,70,10042B660,Cash,0.31
14976,70,10042B660,Cash,1.00
15104,70,10042B660,Cash,1.51
11258,70,10042B660,Credit Card,1.53
...,...,...,...,...
5632,376418,81BEE0220,Cash,532.13
937,223057,81BF9C720,ACH,2705.17
8253,219687,81BF9EF30,Credit Card,16.44
4162,262366,81C039AF0,Credit Card,3.28


In [225]:
#4. Accounts that match the scatter-gather pattern and where the source account has transferred to more than 5 distinct accounts.

accounts_df = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
account_pairs_df = accounts_df.merge(accounts_df, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]).rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]
account_pairs_df = account_pairs_df.groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
account_pairs_df = account_pairs_df[(account_pairs_df["size"] > 5)]


from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts

,Bank,Account


In [226]:
#Conversion dates of period [2022-09-01, 2022-09-05] with base USD
#Bitcoin rates taken from investing.com
#Rest of currencies from api.frankfurter.dev
conversion_rates_records = np.rec.array([
           ('2022/09/01', 1.4644, 5.1805, 1.314 , 0.97999, 6.9   , 1.0002, 0.86272, 3.3535, 79.543, 139.34, 20.189, 60.367, 3.75, 1.,  19793.1),
           ('2022/09/02', 1.4691, 5.2035, 1.3141, 0.98175, 6.9035, 1.0011, 0.86468, 3.3755, 79.719, 140.11, 20.085, 60.427, 3.75, 1., 199999. ),
           ('2022/09/03', 1.4691, 5.2056, 1.3138, 0.98207, 6.9046, 1.0013, 0.86478, 3.3791, 79.75 , 140.17, 20.081, 60.471, 3.75, 1.,  19831.4),
           ('2022/09/04', 1.4695, 5.2082, 1.3139, 0.98219, 6.9047, 1.0013, 0.8649 , 3.3815, 79.754, 140.22, 20.084, 60.461, 3.75, 1.,  19952.7),
           ('2022/09/05', 1.4722, 5.1786, 1.3142, 0.98273, 6.9216, 1.0068, 0.86813, 3.4006, 79.816, 140.49, 20.018, 60.737, 3.75, 1.,  20126.1)],
          dtype=[ ('Date', 'O'), ('Australian Dollar', '<f8'), ('Brazil Real', '<f8'), ('Canadian Dollar', '<f8'), ('Swiss Franc', '<f8'), ('Yuan', '<f8'), ('Euro', '<f8'), ('UK Pound', '<f8'), ('Shekel', '<f8'), ('Rupee', '<f8'), ('Yen', '<f8'), ('Mexican Peso', '<f8'), ('Ruble', '<f8'), ('Saudi Riyal', '<f8'), ('US Dollar', '<f8'), ('Bitcoin', '<f8')])
conversion_rates_df = pd.DataFrame.from_records(conversion_rates_records)
conversion_rates_df = conversion_rates_df.set_index("Date")

In [227]:
#5. Count of transactions of period [2022-09-01, 2022-09-05] with type Wire or ACH, having converted amount for that day less than USD 1.
trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= '2022/09/01') & (trans_df["Timestamp"] <= '2022/09/06')]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[(trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df['Amount'] = trans_sept_1st_wire_or_ach_converted_df.apply(lambda row: row['Amount Paid'] / conversion_rates_df[row['Payment Currency']][row["Timestamp"].split(" ")[0]], axis=1)
trans_sept_1st_wire_or_ach_filtered = trans_sept_1st_wire_or_ach_converted_df[trans_sept_1st_wire_or_ach_converted_df['Amount'] < 1.0]
print("SIZE:", trans_sept_1st_wire_or_ach_filtered.shape[0])

SIZE: 164


In [228]:
from pandas.testing import assert_frame_equal
result_q1 = pd.read_csv(f"./output/q1_output_{INPUT_ID}.csv")
q1_comp1=low_profile_transactions.sort_values(by=["From Bank", "Account"], ascending=True).reset_index(drop=True)
q1_comp2=result_q1.sort_values(by=["From Bank", "Account"], ascending=True).reset_index(drop=True).round(2)

assert_frame_equal(q1_comp1, q1_comp2)

In [229]:
result_q2 = pd.read_csv(f"./output/q2_output_{INPUT_ID}.csv")
q2_comp1=max_amount_bank.sort_values(by=["Amount Paid", "Account"], ascending=True).reset_index(drop=True)
q2_comp2=result_q2.sort_values(by=["Amount Paid", "Account"], ascending=True).reset_index(drop=True).round(2)

assert_frame_equal(q2_comp1[q2_comp2.columns], q2_comp2)

In [230]:
result_q3 = pd.read_csv(f"./output/q3_output_{INPUT_ID}.csv")
q3_comp1=lower_trans_usd_sept_2nd_with_avg_df.sort_values(by=["From Bank", "Account", "Payment Format", "Amount Paid"], ascending=True).reset_index(drop=True)
q3_comp2=result_q3.sort_values(by=["From Bank", "Account", "Payment Format", "Amount Paid"], ascending=True).reset_index(drop=True).round(2)
df = q3_comp1[q3_comp2.columns].merge(q3_comp2, on=q3_comp2.columns.tolist(), how='outer', suffixes=['', '_'], indicator=True)
print(df[df['_merge'] != 'both'])
assert_frame_equal(q3_comp1[q3_comp2.columns], q3_comp2)

Empty DataFrame
Columns: [From Bank, Account, Amount Paid, Payment Format, _merge]
Index: []


In [231]:
df = pd.read_csv(f'./output/q5_output_{INPUT_ID}.csv', header=None)
valor_unico = df.iloc[0, 0]
print(trans_sept_1st_wire_or_ach_filtered.shape[0]==valor_unico) 

True
